### AmadeusAPI

In [14]:
import sys
import os

# Check Python version
print(f"Python Version: `{sys.version}`")  # Detailed version info
print(f"Base Python location: `{sys.base_prefix}`")
print(f"Current Environment location: `{os.path.basename(sys.prefix)}`", end='\n\n')

Python Version: `3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]`
Base Python location: `C:\Users\LMT\AppData\Local\Programs\Python\Python312`
Current Environment location: `.venv`



In [19]:
import requests
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
API_KEY = os.getenv('AMADEUS_API_KEY')
API_SECRET = os.getenv('AMADEUS_API_SECRET')

In [20]:
class FlightDealHunter:
    def __init__(self, api_key, api_secret, use_production=False):
        self.api_key = api_key
        self.api_secret = api_secret
        
        # Choose environment
        if use_production:
            self.base_url = "https://api.amadeus.com"
            print("Using PRODUCTION environment (real-time data)")
        else:
            self.base_url = "https://test.api.amadeus.com"
            print("Using TEST environment (cached data)")
        
        self.token = None
        # self.load_transit_rules()
    
    def get_access_token(self):
        """Get OAuth2 token"""
        url = f"{self.base_url}/v1/security/oauth2/token"
        
        headers = {
            "Content-Type": "application/x-www-form-urlencoded"
        }
        
        data = {
            "grant_type": "client_credentials",
            "client_id": self.api_key,
            "client_secret": self.api_secret
        }
        
        response = requests.post(url, headers=headers, data=data)
        self.token = response.json()["access_token"]
        return self.token
    
    def search_flights(self, origin, destination, departure_date, return_date, max_results=3):
        """Search flights using v2 API"""
        url = f"{self.base_url}/v2/shopping/flight-offers"
        
        headers = {
            "Authorization": f"Bearer {self.token}"
        }
        
        params = {
            "originLocationCode": origin,
            "destinationLocationCode": destination,
            "departureDate": departure_date,
            "returnDate": return_date,
            "adults": 1,
            "currencyCode": "GBP",
            "max": max_results
        }
        
        response = requests.get(url, headers=headers, params=params)
        return response.json()


In [23]:
scanner = FlightDealHunter(API_KEY, API_SECRET, use_production=False)

scanner.get_access_token()

results = scanner.search_flights(
    origin="LON",
    destination="SGN",
    departure_date="2025-12-20",
    return_date="2026-01-10"
)
results

Using TEST environment (cached data)


{'errors': [{'status': 400,
   'code': 425,
   'title': 'INVALID DATE',
   'detail': 'Date/Time is in the past',
   'source': {}}]}

In [31]:
API_KEY = os.getenv('AMADEUS_API_KEY')
API_SECRET = os.getenv('AMADEUS_API_SECRET')

In [44]:
from amadeus import Client, ResponseError

amadeus = Client(
    client_id=API_KEY,
    client_secret=API_SECRET,
    # hostname="production"  # omit for test
)

response = amadeus.shopping.flight_offers_search.get(
    originLocationCode="LON",
    destinationLocationCode="SGN",
    departureDate="2026-03-07",
    returnDate="2026-03-09",
    adults=1,
    currencyCode="GBP",
    max=3,
)
results = response.data


results


ClientError: [400]


In [38]:
amadeus = Client(
    client_id=API_KEY,
    client_secret=API_SECRET,
    # log_level='debug'  # shows full error details
)

try:
    response = amadeus.shopping.flight_offers_search.get(
        originLocationCode='MAD',
        destinationLocationCode='ATH',
        departureDate='2026-06-01',
        adults=1
    )
    print(response.data)
except ResponseError as error:
    print(f"Status: {error.response.status_code}")
    print(f"Body: {error.response.body}")


[{'type': 'flight-offer', 'id': '1', 'source': 'GDS', 'instantTicketingRequired': False, 'nonHomogeneous': False, 'oneWay': False, 'isUpsellOffer': False, 'lastTicketingDate': '2026-05-07', 'lastTicketingDateTime': '2026-05-07', 'numberOfBookableSeats': 9, 'itineraries': [{'duration': 'PT7H', 'segments': [{'departure': {'iataCode': 'MAD', 'terminal': '2', 'at': '2026-06-01T17:10:00'}, 'arrival': {'iataCode': 'AMS', 'at': '2026-06-01T19:45:00'}, 'carrierCode': 'KL', 'number': '1506', 'aircraft': {'code': '73J'}, 'operating': {'carrierCode': 'KL'}, 'duration': 'PT2H35M', 'id': '3', 'numberOfStops': 0, 'blacklistedInEU': False}, {'departure': {'iataCode': 'AMS', 'at': '2026-06-01T21:00:00'}, 'arrival': {'iataCode': 'ATH', 'at': '2026-06-02T01:10:00'}, 'carrierCode': 'KL', 'number': '1957', 'aircraft': {'code': '32Q'}, 'operating': {'carrierCode': 'KL'}, 'duration': 'PT3H10M', 'id': '4', 'numberOfStops': 0, 'blacklistedInEU': False}]}], 'price': {'currency': 'EUR', 'total': '115.60', 'base